In [1]:
# ============================================================
# FINAL SCALED_EI01 — EXPORT HUB (ELECTRICITY ONLY)
# Offshore wind + HVDC export + Battery
# ============================================================

import pypsa
import numpy as np
from pathlib import Path

# ------------------------------------------------------------
# PATHS
# ------------------------------------------------------------
root = Path(r"C:\Users\franc\pypsa\pypsa-eur")
base_fn = root / "resources" / "networks" / "base_s_50_elec.nc"
out_fn  = root / "results" / "networks" / "EI01_export_hub.nc"

n = pypsa.Network(base_fn)
print("Loaded base network:", base_fn.name)

# ------------------------------------------------------------
# COUNTRIES
# ------------------------------------------------------------
EI_COUNTRIES      = ["DK", "DE", "NL", "NO", "SE"]
PROFILE_COUNTRIES = ["DK", "DE", "NL", "NO"] #Perfil do vento (north sea)

# ------------------------------------------------------------
# NETCDF SAFETY
# ------------------------------------------------------------
if "color" in n.carriers.columns:
    n.carriers["color"] = n.carriers["color"].astype(str)

# ------------------------------------------------------------
# DISPATCH-ONLY
# ------------------------------------------------------------
for comp in ["generators", "links", "lines", "storage_units", "stores"]:
    if hasattr(n, comp):
        if "p_nom_extendable" in getattr(n, comp).columns:
            getattr(n, comp)["p_nom_extendable"] = False

if "s_nom_extendable" in n.lines.columns:
    n.lines["s_nom_extendable"] = False

print("✓ Dispatch-only mode enforced")

# ============================================================
# EXPORT HUB
# ============================================================

EI_BUS = "EI01_bus"

if EI_BUS not in n.buses.index:
    n.add("Bus", EI_BUS, carrier="AC", country="EI", x=9.0, y=56.5)

print("✓ Export Hub bus added")

# ------------------------------------------------------------
# AC CONNECTION TO MAINLAND (CRITICAL)
# ------------------------------------------------------------
ref_bus = (
    n.buses[n.buses.country.isin(EI_COUNTRIES)]
    .index[0]
)

n.add(
    "Line",
    name="EI01_AC_tie",
    bus0=EI_BUS,
    bus1=ref_bus,
    s_nom=20_000,   # MW
    x=0.01,
    r=0.001,
)

print("✓ AC tie-line added")

# ------------------------------------------------------------
# REQUIRED CARRIERS (NO H2)
# ------------------------------------------------------------
REQUIRED_CARRIERS = [
    "offwind-ac",
    "battery",
    "DC",
    "load_shedding",
]

for c in REQUIRED_CARRIERS:
    if c not in n.carriers.index:
        n.carriers.loc[c] = {col: 0 for col in n.carriers.columns}

print("✓ Required carriers ensured")

# ------------------------------------------------------------
# OFFSHORE WIND PROFILE (NORTH SEA)
# ------------------------------------------------------------
offwind_mask = (
    n.generators.carrier.str.contains("offwind")
    & n.generators.bus.map(n.buses.country).isin(PROFILE_COUNTRIES)
)

p_max_pu_ns = (
    n.generators_t.p_max_pu.loc[:, offwind_mask]
    .mean(axis=1)
    .clip(0.0, 1.0)
)

# ------------------------------------------------------------
# OFFSHORE WIND — 15 GW
# ------------------------------------------------------------
n.add(
    "Generator",
    "EI01_offshore_wind",
    bus=EI_BUS,
    carrier="offwind-ac",
    p_nom=15_000,
    marginal_cost=0.0,
)

n.generators_t.p_max_pu["EI01_offshore_wind"] = p_max_pu_ns.values
print("✓ Offshore wind added (15 GW)")

# ------------------------------------------------------------
# HVDC LINKS — 15 GW TOTAL (3 GW × 5 COUNTRIES)
# ------------------------------------------------------------
HVDC_CAP = 3_000  # MW per country

for c in EI_COUNTRIES:
    loads_c = n.loads[n.loads.bus.map(n.buses.country) == c]
    main_bus = loads_c.groupby("bus").p_set.sum().idxmax()

    n.add(
        "Link",
        f"EI01_HVDC_{c}",
        bus0=EI_BUS,
        bus1=main_bus,
        carrier="DC",
        p_nom=HVDC_CAP,
        efficiency=0.98,
        marginal_cost=1.0,
    )

print("✓ HVDC export links added (15 GW total)")

# ------------------------------------------------------------
# BATTERY — .5 GWh
# ------------------------------------------------------------
n.add(
    "StorageUnit",
    "EI01_battery",
    bus=EI_BUS,
    carrier="battery",
    p_nom=1_000,      # MW
    max_hours=0.5,    # → 0.5 GWh
    efficiency_store=0.95,
    efficiency_dispatch=0.95,
    cyclic_state_of_charge=False,
)


print("✓ Battery added")

# ------------------------------------------------------------
# LOAD SHEDDING (SAFETY)
# ------------------------------------------------------------
for b in n.buses.index:
    g = f"load_shed_{b}"
    if g not in n.generators.index:
        n.add(
            "Generator",
            g,
            bus=b,
            carrier="load_shedding",
            p_nom=1e6,
            marginal_cost=10_000.0,
        )

print("✓ Load shedding ensured")

# ------------------------------------------------------------
# FINAL NETCDF SAFETY
# ------------------------------------------------------------
for col in ["color", "nice_name"]:
    if col in n.carriers.columns:
        n.carriers[col] = n.carriers[col].astype(str).fillna("")

# ------------------------------------------------------------
# SAVE
# ------------------------------------------------------------
out_fn.parent.mkdir(parents=True, exist_ok=True)
n.export_to_netcdf(out_fn)

print("\n✓ EI01 Export Hub network saved:")
print(out_fn)


INFO:pypsa.io:Imported network base_s_50_elec.nc has buses, carriers, generators, lines, links, loads, storage_units, stores


Loaded base network: base_s_50_elec.nc
✓ Dispatch-only mode enforced
✓ Export Hub bus added
✓ AC tie-line added
✓ Required carriers ensured
✓ Offshore wind added (15 GW)
✓ HVDC export links added (15 GW total)
✓ Battery added
✓ Load shedding ensured


INFO:pypsa.io:Exported network 'EI01_export_hub.nc' contains: loads, links, generators, stores, lines, buses, carriers, storage_units



✓ EI01 Export Hub network saved:
C:\Users\franc\pypsa\pypsa-eur\results\networks\EI01_export_hub.nc
